In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial import cKDTree

# --- Example data ---
# Replace these with your real DataFrames
df_wd = pd.DataFrame({
    "name": ["Hundpark A", "Hundpark B", "Hundpark C"],
    "lat": [59.332, 59.345, 59.31],
    "lon": [18.064, 18.03, 18.09],
})
df_osm = pd.DataFrame({
    "name": ["OSM Hund A", "OSM Hund B"],
    "lat": [59.333, 59.346],
    "lon": [18.065, 18.031],
})

# --- Convert to GeoDataFrames (EPSG:4326 -> 3857 for meters) ---
gdf_wd = gpd.GeoDataFrame(df_wd, geometry=gpd.points_from_xy(df_wd.lon, df_wd.lat), crs="EPSG:4326").to_crs(3857)
gdf_osm = gpd.GeoDataFrame(df_osm, geometry=gpd.points_from_xy(df_osm.lon, df_osm.lat), crs="EPSG:4326").to_crs(3857)

# --- Use KDTree to find nearest OSM point for each Wikidata point ---
osm_coords = list(zip(gdf_osm.geometry.x, gdf_osm.geometry.y))
tree = cKDTree(osm_coords)

wd_coords = list(zip(gdf_wd.geometry.x, gdf_wd.geometry.y))
distances, indices = tree.query(wd_coords, k=1)

gdf_wd["nearest_osm"] = [gdf_osm.iloc[i]["name"] for i in indices]
gdf_wd["distance_m"] = distances

# --- Filter points where distance > 300m ---
missing_in_osm = gdf_wd[gdf_wd["distance_m"] > 300]
missing_in_osm


,name,lat,lon,geometry,nearest_osm,distance_m
2,Hundpark C,59.31,18.09,POINT (2013769.588 8247693.603),OSM Hund A,5738.169346


In [3]:
import folium 
m = folium.Map(location=[59.494362874953964, 14.877945359768825], zoom_start=6)


In [7]:
import folium
m = folium.Map(location=[59.494362874953964, 14.877945359768825], zoom_start=6)


for _, row in missing_in_osm.iterrows():
    print(row)
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=7,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.7,
        popup=f"🐾 {row['name']}<br>Avstånd till närmaste OSM: {row['distance_m']:.0f} m<br>Närmast: {row['nearest_osm']}"
    ).add_to(m)
m

name                                            Hundpark C
lat                                                  59.31
lon                                                  18.09
geometry       POINT (2013769.588450319 8247693.602747885)
nearest_osm                                     OSM Hund A
distance_m                                     5738.169346
Name: 2, dtype: object
